In [19]:
# Core Python imports
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore",message="In a future version of xarray the default value for join",category=FutureWarning)
warnings.filterwarnings("ignore",message="In a future version of xarray the default value for compat",category=FutureWarning)
warnings.filterwarnings("ignore",message="Data requested at a higher resolution than available")
warnings.filterwarnings("ignore",message="Importing `spectral_angle_mapper` from `torchmetrics.functional`",category=FutureWarning)
# Scientific standard imports
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# PyEarthTools imports including NCI cached data
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
from pyearthtools.data.time import Petdt

my_site = 'site_archive_nci'  # set this to 'site_archive_nci', 'site_archive_jasmin' or 'site_archive_met_office'
import importlib
_ = importlib.import_module(my_site)

In [41]:
# We specify the date, hour, and minute for querying data
# CURRENTLY SET TO ONE MONTH!!!
date = '20200105T0000'
start_date = '20200101T00'
end_date = '20200201T00'
sat_timestep = '10 minutes'
bar_timestep = "1 hour" 

In [35]:
n_prior_sat = 12
n_prior_bar = n_prior_sat // 6
n_post = 1

In [36]:
train_split = petpipe.iterators.DateRange(start_date, end_date, interval=bar_timestep)

In [37]:
himawar_vars = [
    'surface_global_irradiance',
    'solar_elevation'
]
himawari = petdata.archive.Himawari(himawar_vars)


sat_pipe = petpipe.Pipeline(
    himawari,
    petdata.transform.region.Bounding(-35, -28.5, 145, 151.5),
    petpipe.operations.xarray.conversion.ToNumpy(),
    iterator=train_split,
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

In [38]:
barra_convective_vars = [
    "RH24mean"
]
barra_conv = petdata.archive.BARRA_V2(barra_convective_vars, domain_id='AUST-11', frequency='1hr')

bar_pipe = petpipe.Pipeline(
    barra_conv,
    petdata.transforms.coordinates.Drop("crs"),
    petdata.transform.region.Bounding(-35, -28.5, 145, 151.5),
    petpipe.operations.xarray.conversion.ToNumpy(),
    iterator=train_split,
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

In [39]:
save_dir = Path("/scratch/er8/cd3022/CPDiT/stats/")

In [40]:
%%time
bar_samples = np.stack([bar_pipe[i] for i in train_split])

mean_path = save_dir / "mean.npy"
np.save(mean_path, np.mean(bar_samples, axis=0))

std_path = save_dir / "std.npy"
np.save(std_path, np.std(bar_samples, axis=0))

CPU times: user 8.12 s, sys: 614 ms, total: 8.73 s
Wall time: 10.6 s


In [42]:
normaliser = petpipe.operations.numpy.normalisation.Deviation(
    mean=mean_path,
    deviation=std_path,
    expand=False
)

In [48]:
caching_step = petpipe.modifications.Cache(
    save_dir / "cache",
    pattern_kwargs={'extension': 'npy'},
)

In [49]:
bar_pipe_nomred = petpipe.Pipeline(
    bar_pipe,
    normaliser,
)

In [52]:
bar_pipe_nomred["20200101T0200"]

np.float64(0.5400364133190755)